In [1]:
import pandas as pd
from pathlib import Path
from typing import List, Set, Tuple
import glob

In [2]:
ONN_DIR = Path(
    "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/ONN/initial/ONNResults"
)
PAIRS: List[Tuple[str, str, str]] = [
    #species_1 ,      species_2 ,                                  match_dir
    ("human",            "mice",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Homo_sapiens.GRCh38.pep.all_Mus_musculus.GRCm39.pep.all"),
    ("chimpanzees",      "gorillas",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Pan_troglodytes.Pan_tro_3.0.pep.all_Gorilla_gorilla.gorGor4.pep.all"),
    ("crab_eating_macaque", "mice",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Macaca_fascicularis.Macaca_fascicularis_6.0.pep.all_Mus_musculus.GRCm39.pep.all"),
    ("crab_eating_macaque", "rhesus_macaque",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Macaca_fascicularis.Macaca_fascicularis_6.0.pep.all_Macaca_mulatta.Mmul_10.pep.all"),
    ("gorillas",          "rhesus_macaque",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Gorilla_gorilla.gorGor4.pep.all_Macaca_mulatta.Mmul_10.pep.all"),
    ("human",             "chimpanzees",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Homo_sapiens.GRCh38.pep.all_Pan_troglodytes.Pan_tro_3.0.pep.all"),
    ("microcebus",        "human",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Microcebus_murinus.Mmur_3.0.pep.all_Homo_sapiens.GRCh38.pep.all"),
    ("mice",              "frog",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Mus_musculus.GRCm39.pep.all_Xenopus_tropicalis.UCB_Xtro_10.0.pep.all"),
    ("mice",              "pig",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Mus_musculus.GRCm39.pep.all_Sus_scrofa.Sscrofa11.1.pep.all"),
    ("rhesus_macaque",    "human",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Macaca_mulatta.Mmul_10.pep.all_Homo_sapiens.GRCh38.pep.all"),
    ("rhesus_macaque",    "commonmarmosets",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Macaca_mulatta.Mmul_10.pep.all_Callithrix_jacchus.mCalJac1.pat.X.pep.all"),
    ("human",    "pig",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Homo_sapiens.GRCh38.pep.all_Sus_scrofa.Sscrofa11.1.pep.all"),
    ("zebrafish",    "frog",
     "/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/LM_O2O/Species_mapping/Results/Danio_rerio.GRCz11.pep.all_Xenopus_tropicalis.UCB_Xtro_10.0.pep.all"),
    
]

In [3]:
def global_greedy_match(
    corr_mat: pd.DataFrame,
    pairs: pd.DataFrame,
    col1: str,
    col2: str,
) -> pd.DataFrame:
    """相关系数全局 greedy → 严格 1-to-1"""
    df = pairs.copy()

    # —— 1. 计算 correlation —— #
    def fetch(row) -> float:
        try:
            return corr_mat.at[row[col1], row[col2]]
        except KeyError:
            return float("nan")

    df["correlation"] = df[[col1, col2]].apply(fetch, axis=1)
    df = df.dropna(subset=["correlation"])

    # —— 2. greedy —— #
    used_f, used_s, selected = set(), set(), []
    sorter = df.sort_values(
        by=["correlation", col1, col2], ascending=[False, True, True], kind="mergesort"
    )
    for _, row in sorter.iterrows():
        f, s = row[col1], row[col2]
        if f not in used_f and s not in used_s:
            selected.append(row)
            used_f.add(f)
            used_s.add(s)

    return pd.DataFrame(selected, columns=[col1, col2, "correlation"])



In [4]:
for sp1, sp2, match_dir in PAIRS:
    print(f"\n=== {sp1} ↔ {sp2} ===")
    match_dir = Path(match_dir)

    # ---------- 2.1 读取 ONN 候选 ----------
    onn_file = ONN_DIR / f"{sp1}_{sp2}.csv"
    if not onn_file.exists():
        print(f"  ❌ ONN 结果缺失：{onn_file.name}，跳过")
        continue
    onn_df = pd.read_csv(onn_file)

    # ---------- 2.2 找到 correlation / embedding 文件 ----------
    try:
        corr_file = next(match_dir.glob("*gene_embeddings_correlation_matrix.csv"))
    except StopIteration:
        print("  ❌ 未找到相关系数矩阵文件，跳过")
        continue

    embed1 = match_dir / "gene_embedding_1_matrix.csv"
    embed2 = match_dir / "gene_embedding_2_matrix.csv"
    if not embed1.exists() or not embed2.exists():
        print("  ❌ gene_embedding_1/2_matrix.csv 缺失，跳过")
        continue

    # ---------- 2.3 读取矩阵 ----------
    corr_mat = pd.read_csv(corr_file, index_col=0)
    gene1 = pd.read_csv(embed1, nrows=0).columns.tolist()[1:]  # 跳过第一列空 label
    gene2 = pd.read_csv(embed2, nrows=0).columns.tolist()[1:]
    corr_mat.index = gene1
    corr_mat.columns = gene2

    # ---------- 2.4 全局 greedy ----------
    col1, col2 = onn_df.columns[:2]
    matched = global_greedy_match(corr_mat, onn_df, col1, col2)
    print(f"  ✅ 得到 1-to-1 配对 {matched.shape[0]} 对")

    # ---------- 2.5 保存新结果 ----------
    (ONN_DIR).mkdir(exist_ok=True)  # 保底
    new_path = ONN_DIR / f"{sp1}_{sp2}_LM_O2O.csv"
    matched.to_csv(new_path, index=False)
    print(f"  新结果已保存：{new_path.name}")


=== zebrafish ↔ frog ===
  ✅ 得到 1-to-1 配对 12761 对
  新结果已保存：zebrafish_frog_o2o_all.csv
  ⚠ 未找到旧版 o2o 文件，跳过差异比对
  ✅ 未发现重复基因
